# Merge Retriever and Context Reorder | Lost in Middle Phenomenon

## Install required libraries

In [ ]:
!pip install -qU \
langchain \
langchain-core \
langchain-community \
chromadb \
huggingface_hub \
sentence-transformers \
pypdf \
groq \
tiktoken \
opentelemetry-api \
opentelemetry-sdk \
opentelemetry-proto \
opentelemetry-exporter-otlp-proto-common

In [ ]:
!pip install -qU langchain-groq

## Load Data

In [ ]:
from langchain.document_loaders import PyPDFLoader

In [ ]:
loader = PyPDFLoader("/content/RS1.pdf")

In [ ]:
document = loader.load()

In [ ]:
print(len(document))

In [ ]:
loader_rs2 = PyPDFLoader("/content/RS2.pdf")

In [ ]:
document_rs2 = loader_rs2.load()

In [ ]:
print(len(document_rs2))

## Create Text Splitter for chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 500, chunk_overlap = 100)

In [ ]:
text_rs1 = text_splitter.split_documents(document)

In [ ]:
text_rs2 = text_splitter.split_documents(document_rs2)

In [ ]:
print(len(text_rs1))

## Load the Embedding Model to Convert Data into Vector

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings, HuggingFaceBgeEmbeddings

In [ ]:
hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
hf_bge_embeddings = HuggingFaceEmbeddings(model_name = "BAAI/bge-large-en")

In [ ]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
huggingface_embeddings = HuggingFaceEmbeddings()

## Ingest data into Chroma Database

In [ ]:
from langchain.vectorstores import Chroma
import chromadb

In [ ]:
import os
os.getcwd()

In [ ]:
CURRENT_DIR = os.path.dirname(os.path.abspath("."))

In [ ]:
CURRENT_DIR

In [ ]:
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")

In [ ]:
DB_DIR

In [ ]:
client_settings = chromadb.config.Settings(
    is_permissions = True,
    persist_directory = DB_DIR,
    anonymized_telemetry = False,
)

In [ ]:
rs1_vectorstore = Chroma.from_documents(documents = text_rs1,
                                        embedding = hf_embeddings,
                                        client_settings = client_settings,
                                        collection_name = "rs1",
                                        collection_metadata = {"hnsw:space": "cosine"},
                                        persist_directory = DB_DIR)

In [ ]:
rs2_vectorstore = Chroma.from_documents(documents = text_rs2,
                                        embedding = hf_bge_embeddings,
                                        client_settings = client_settings,
                                        collection_name = "rs2",
                                        collection_metadata = {"hnsw:space": "cosine"},
                                        persist_directory = DB_DIR)

## Create Retriever

In [ ]:
retriever_rs1 = rs1_vectorstore.as_retriever(search_type = "similarity",  search_kwargs={"k": 5})

In [ ]:
retriever_rs2 = rs2_vectorstore.as_retriever(search_type = "similarity", search_kwargs = {"k": 5 })

## Merge both Retriever
## This is also called lord of retriever (LOTR)

In [ ]:
!pip install langchain-core

In [ ]:
from langchain.retrievers.merger_retriever import MergerRetriever

In [ ]:
lotr = MergerRetriever(retrievers = [retriever_rs1, retriever_rs2])

In [ ]:
for chunks in lotr.invoke("What is RAG?"):
  print(chunks.page_content)

## Create Pipeline

In [ ]:
from langchain_community.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
    LongContextReorder,
)
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain_community.retrievers import ContextualCompressionRetriever

In [ ]:
from re import search
filter = EmbeddingsRedundantFilter(embeddings=hf_bge_embeddings)
reordering = LongContextReorder()
pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])
compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=lotr,search_kwargs={"k": 3, "include_metadata": True}
)

## Load the model from HuggingFace

In [ ]:
!pip install llama-cpp-python

In [ ]:
from langchain.llms import LlamaCpp
llms = LlamaCpp(streaming=True,
                   model_path="/content/drive/MyDrive/zephyr-7b-beta.Q4_K_M.gguf",
                   max_tokens = 1500,
                   temperature=0.75,
                   top_p=1,
                   gpu_layers=0,
                   stream=True,
                   verbose=True,n_threads = int(os.cpu_count()/2),
                   n_ctx=4096)

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
qa = RetrievalQA.from_chain_type(
      llm=llms,
      chain_type="stuff",
      retriever = compression_retriever_reordered,
      return_source_documents = True
)

In [ ]:
query ="what is RAG?"
results = qa(query)
print(results['result'])
print(results["source_documents"])